In [19]:
import pandas as pd
import duckdb
import os
import os, json
from uuid import uuid4
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

In [34]:
conn = duckdb.connect("/srv/data/grela/grela_v0-3.duckdb")
#conn.execute("CREATE TABLE works AS SELECT * FROM works_df")

In [21]:
# make a simple query to extract all works or a subset of works
query = """
SELECT *
FROM works
/*
    alternatively uncomment this:
    WHERE grela_id LIKE 'vulgate_tlg0031%' OR grela_id LIKE 'vulgate_tlg0527%';
*/
"""
works_df = conn.execute(query).fetchdf()

In [22]:
conn.execute("""
ALTER TABLE works ADD COLUMN IF NOT EXISTS textsource VARCHAR;
""")

In [23]:
sentences_info = conn.execute(f"""
        PRAGMA table_info(sentences)
    """).fetchall()
sentences_info

[(0, 'sentence_id', 'VARCHAR', False, None, False),
 (1, 'grela_id', 'VARCHAR', False, None, False),
 (2, 'position', 'INTEGER', False, None, False),
 (3, 'text', 'VARCHAR', False, None, False),
 (4, 'subwork_id', 'VARCHAR', False, None, False)]

In [24]:
tokens_info = conn.execute(f"""
        PRAGMA table_info(tokens)
    """).fetchall()
tokens_info

[(0, 'sentence_id', 'VARCHAR', False, None, False),
 (1, 'grela_id', 'VARCHAR', False, None, False),
 (2, 'token_text', 'VARCHAR', False, None, False),
 (3, 'lemma', 'VARCHAR', False, None, False),
 (4, 'pos', 'VARCHAR', False, None, False),
 (5, 'char_start', 'INTEGER', False, None, False),
 (6, 'char_end', 'INTEGER', False, None, False),
 (7, 'token_id', 'BIGINT', False, None, False),
 (8, 'ref', 'JSON', False, None, False)]

In [25]:
result = conn.execute(f"""
        PRAGMA table_info(tokens)
    """).fetchall()

    # Extract column names
columns = [row[1] for row in result]

if "ref" not in columns:
    conn.execute(f"ALTER TABLE tokens ADD COLUMN ref JSON")
    conn.execute(f"UPDATE tokens SET ref = '{{}}'")


In [26]:
import os, json, pickle
from pathlib import Path
import pandas as pd

def process_single_pickle(pickle_path: Path, grela_prefix: str, jsonize_ref: bool = True):
    """Load ONE .pickle file and return sentences_df, tokens_df for that grela_id."""
    base_id = pickle_path.stem
    grela_id = f"{grela_prefix}_{base_id}"

    with open(pickle_path, "rb") as f:
        sents_data = pickle.load(f)

    sentences = []
    tokens = []

    for sent in sents_data:
        # works with either 4-tuple or 5-tuple (if cts_source is appended)
        work_id, pos, text, token_data = sent[:4]
        sentence_id = f"{grela_id}_{pos}"
        sentences.append([sentence_id, grela_id, pos, text, ""])  # subwork_id empty for now

        for tok in token_data:
            tok_text, lemma, pos_tag, ref, char_start, char_end = tok[:6]
            if jsonize_ref and isinstance(ref, dict):
                ref_val = json.dumps(ref, ensure_ascii=False)
            else:
                # DuckDB doesn’t like pandas “object” dicts much; JSON string is faster
                ref_val = ref if isinstance(ref, (str, type(None))) else None

            tokens.append([
                sentence_id,
                grela_id,
                tok_text,
                lemma,
                pos_tag,
                int(char_start),
                int(char_end),
                None,         # token_id (optional, keep None)
                ref_val,      # JSON-ified ref for speed
            ])

    sentences_df = pd.DataFrame(
        sentences,
        columns=["sentence_id", "grela_id", "position", "text", "subwork_id"],
    )
    # Optional: enforce dtypes for speed
    sentences_df = sentences_df.astype({
        "sentence_id": "string",
        "grela_id": "string",
        "position": "int32",
        "text": "string",
        "subwork_id": "string",
    })

    tokens_df = pd.DataFrame(
        tokens,
        columns=["sentence_id", "grela_id", "token_text", "lemma", "pos",
                 "char_start", "char_end", "token_id", "ref"],
    )
    tokens_df = tokens_df.astype({
        "sentence_id": "string",
        "grela_id": "string",
        "token_text": "string",
        "lemma": "string",
        "pos": "string",
        "char_start": "int32",
        "char_end": "int32",
        # token_id stays nullable; ref is JSON string or None
    })

    return grela_id, sentences_df, tokens_df

In [42]:
%%time
SOURCE_MAP = {
    "/srv/data/greek/exprecce_sentences_2025-08/": "exprecce",
    "/srv/data/greek/glaux_sentences_2025-08/":    "glaux",
    "/srv/data/greek/oga_sentences_2025-08/":      "oga",
}

data_sources = [
    ("/srv/data/greek/exprecce_sentences_2025-08/", "lagt"),
    ("/srv/data/greek/glaux_sentences_2025-08/",    "lagt"),
    ("/srv/data/greek/oga_sentences_2025-08/",      "lagt"),
]

seen_grela_ids = set()

conn.execute("BEGIN;")
try:
    for dir_path, prefix in data_sources:
        textsource = SOURCE_MAP[dir_path]
        for i, pickle_path in enumerate(sorted(Path(dir_path).glob("*.pickle")), start=1):
            base_id = pickle_path.stem
            grela_id = f"{prefix}_{base_id}"

            if grela_id in seen_grela_ids:
                continue

            grela_id_check, sents_df, toks_df = process_single_pickle(pickle_path, prefix, jsonize_ref=True)
            assert grela_id_check == grela_id

            # refresh rows for this grela_id
            conn.execute("DELETE FROM sentences WHERE grela_id = ?", [grela_id])
            conn.execute("DELETE FROM tokens    WHERE grela_id = ?", [grela_id])

            conn.register("temp_sentences", sents_df)
            conn.execute("INSERT INTO sentences SELECT * FROM temp_sentences")
            conn.unregister("temp_sentences")

            conn.register("temp_tokens", toks_df)
            conn.execute("INSERT INTO tokens SELECT * FROM temp_tokens")
            conn.unregister("temp_tokens")

            # Upsert works.textsource, but do NOT overwrite if it’s already set
            conn.execute(
                "UPDATE works SET textsource = ? "
                "WHERE grela_id = ? AND (textsource IS NULL OR textsource = '')",
                [textsource, grela_id],
            )

            # after inserting sentences/tokens for this grela_id
            conn.execute("UPDATE works SET textsource = ? WHERE grela_id = ?", [textsource, grela_id])
            conn.execute(
                "INSERT INTO works (grela_id, textsource) "
                "SELECT ?, ? WHERE NOT EXISTS (SELECT 1 FROM works WHERE grela_id = ?)",
                [grela_id, textsource, grela_id],
            )

            seen_grela_ids.add(grela_id)

            if i % 50 == 0:
                print(f"[{textsource}] processed {i} files from {dir_path}")
    conn.execute("COMMIT;")
except Exception:
    conn.execute("ROLLBACK;")
    raise

[glaux] processed 50 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 100 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 150 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 200 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 250 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 300 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 350 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 400 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 450 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 500 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 550 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 600 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 650 files from /srv/data/greek/glaux_sentences_2025-08/
[glaux] processed 700 files from /srv/d

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 18min 20s, sys: 2min 40s, total: 21min
Wall time: 6min 46s


In [43]:
conn.execute("UPDATE tokens SET token_id = rowid;")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [44]:
conn.execute("""--
UPDATE works AS w
SET    token_count = sub.cnt
FROM (
        SELECT grela_id, COUNT(*) AS cnt
        FROM tokens
        GROUP BY grela_id
     ) AS sub
WHERE w.grela_id = sub.grela_id;
""")

In [45]:
# Query to get table and column information
query = """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    ORDER BY table_name, ordinal_position
"""

# Execute the query and fetch the schema information as a DataFrame
df = conn.execute(query).fetchdf()

# Group the schema details by table
tables = df.groupby("table_name")

# Markdown generation
markdown = "# Database Schema Documentation\n\n"
for table_name, group in tables:
    markdown += f"## Table: `{table_name}`\n\n"
    markdown += "| Column Name     | Data Type    | Is Nullable | Default Value |\n"
    markdown += "|-----------------|-------------|-------------|---------------|\n"

    for _, row in group.iterrows():
        markdown += (
            f"| {row['column_name']} | {row['data_type']} | "
            f"{row['is_nullable']} | {row['column_default'] or 'N/A'} |\n"
        )

    markdown += "\n"  # Add a space between tables


In [46]:
print(markdown)

# Database Schema Documentation

## Table: `sentence_embeddings`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | NO | N/A |
| grela_id | VARCHAR | YES | N/A |
| model | VARCHAR | YES | N/A |
| embedding | JSON | YES | N/A |

## Table: `sentences`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES | N/A |
| position | INTEGER | YES | N/A |
| text | VARCHAR | YES | N/A |
| subwork_id | VARCHAR | YES | N/A |

## Table: `tokens`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES | N/A |
| token_text | VARCHAR | YES | N/A |
| lemma | VARCHAR | YES | N/A |
| pos | VARCHAR | YES | N/A |
| char_start | IN

In [47]:
query = """
    SELECT t.*, w.*
    FROM tokens t
    JOIN works w ON t.grela_id = w.grela_id
    WHERE w.grela_id LIKE 'lagt_tlg0031.tlg001'
"""

gnt_tokens = conn.execute(query).fetchdf()
gnt_tokens.head(10)

,sentence_id,grela_id,token_text,lemma,pos,char_start,char_end,token_id,ref,grela_source,...,noscemus_discipline,title_short,emlap_noscemus_id,place_publication,place_geonames,author_viaf,title_viaf,date_random,token_count,textsource
0,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,βίβλος,βίβλος,NOUN,0,6,349280909,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
1,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,γενέσεως,γένεσις,NOUN,7,15,349280910,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
2,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,Ἰησοῦ,Ἰησοῦς,NOUN,16,21,349280911,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
3,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,Χριστοῦ,Χριστός,NOUN,22,29,349280912,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
4,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,υἱοῦ,υἱός,NOUN,30,34,349280913,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
5,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,Δαυεὶδ,Δαυίδ,NOUN,35,41,349280914,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
6,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,υἱοῦ,υἱός,NOUN,42,46,349280915,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
7,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,Ἀβραάμ,Ἀβραάμ,NOUN,47,53,349280916,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
8,lagt_tlg0031.tlg001_0,lagt_tlg0031.tlg001,.,.,PUNCT,53,54,349280917,"{""div_chapter"": ""1"", ""div_section"": ""1.1""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux
9,lagt_tlg0031.tlg001_1,lagt_tlg0031.tlg001,Ἀβραὰμ,Ἀβραάμ,NOUN,0,6,349280918,"{""div_chapter"": ""1"", ""div_section"": ""1.2""}",lagt,...,None,None,NaN,None,None,NaN,NaN,83.0,19551,glaux


In [48]:
conn.close()

In [7]:
#sents_emlap, tokens_emlap = process_sentences_from_dir(emlap_sents_data_dir, "emlap")

In [8]:
# look at emlap data for testing
#tokens_emlap.sample(10)

,sentence_id,grela_id,token_text,lemma,pos,char_start,char_end,page_idx,textblock_idx
1349552,emlap_100059_3060,emlap_100059,tria,tres,NUM,24,28,[241],[5]
2049492,emlap_100061_2489,emlap_100061,ibid,ibid,ADV,0,4,[108],[5]
2198084,emlap_100063_99,emlap_100063,perpetuitare,perpetuito,VERB,153,165,[16],[19]
3141712,emlap_100039_141,emlap_100039,Mercurio,Mercurius,PROPN,34,42,[20],"[15, 16]"
1937786,emlap_100062_690,emlap_100062,negemus,nego,VERB,95,102,[64],[19]
1504414,emlap_100032_2012,emlap_100032,creationibus,creatio,NOUN,33,45,[146],[21]
1292954,emlap_100053_878,emlap_100053,copia,copia,NOUN,405,410,[133],[19]
3049342,emlap_100015_1257,emlap_100015,theriacalis,theriacalis,ADJ,43,54,[114],[18]
1539254,emlap_100032_4003,emlap_100032,argento,argentum,NOUN,114,121,[324],[23]
979396,emlap_100038_7507,emlap_100038,in,in,ADP,30,32,[575],[25]


In [9]:
# look
sents_emlap.sample(10)

,sentence_id,grela_id,position,text
110291,emlap_100051_3639,emlap_100051,3639,"Ad uenarum humores erassos, lentos, ac pituito..."
49304,emlap_100070_5892,emlap_100070,5892,"Figulina pinguis, sanguis draconis paucus, ter..."
131811,emlap_100061_612,emlap_100061,612,ibid. 91.
124958,emlap_100046_1953,emlap_100046,1953,"Uerum eiusmodi pascua quandoque peiora sunt, q..."
77197,emlap_100042_1078,emlap_100042,1078,"Nec est quod ita mireris, medicinam aliquam in..."
198900,emlap_100015_2627,emlap_100015,2627,"Prima aqua erit clara, & ualet podagrae humidae:"
95195,emlap_100032_1298,emlap_100032,1298,"quia ad formam oui dispositum est, & dicitur s..."
34064,emlap_100013_2815,emlap_100013,2815,"Omni mane & sero de hac aqua impones ad aures,..."
216215,emlap_100011_399,emlap_100011,399,Promissio & diuisio dicendorum de operationib.
94951,emlap_100032_1054,emlap_100032,1054,Hec est huius rei radix ut qui eam addiscere u...


In [ ]:
%%time
sents_nos, tokens_nos = process_sentences_from_dir(noscemus_sents_data_dir, "noscemus")

In [15]:
sents_nos.sample(10)

,sentence_id,grela_id,position,text
743250,noscemus_668522_21910,noscemus_668522,21910,Elacatena quoque salsamento idoneus piscis est...
2565541,noscemus_914304_9018,noscemus_914304,9018,pag. 212.
2661058,noscemus_928147_13394,noscemus_928147,13394,Secunda ibi.
3149822,noscemus_732628_8192,noscemus_732628,8192,Idem inquietis & stolidis ingemus euenit:
9961985,noscemus_901145_78964,noscemus_901145,78964,fiat confectio in rotulis ponderis. 3 8.
3466975,noscemus_631363_13372,noscemus_631363,13372,Calceolario Ueronensi Pharmacopol Ioannis Bapt...
4135971,noscemus_929375_5561,noscemus_929375,5561,Inter sinistrum humerum & Tertiam 2.
8949356,noscemus_906961_21555,noscemus_906961,21555,Harduini in 410.
8237679,noscemus_664561_4548,noscemus_664561,4548,"Pitem attenuat, resoluit, Matt."
2789886,noscemus_835557_24530,noscemus_835557,24530,b. 50.


In [16]:
tokens_nos.sample(10)

,sentence_id,grela_id,token_text,lemma,pos,char_start,char_end,page_idx,textblock_idx
95331451,noscemus_767766_1405,noscemus_767766,camelorum,camelus,NOUN,13,22,None,None
5493374,noscemus_704336_15313,noscemus_704336,impressit,imprimo,VERB,53,62,None,None
99530955,noscemus_918511_10291,noscemus_918511,In,in,ADP,0,2,None,None
91258461,noscemus_906964_7937,noscemus_906964,De,de,ADP,26,28,None,None
92824347,noscemus_655273_3785,noscemus_655273,circulos,circulus,NOUN,82,90,None,None
31477593,noscemus_756874_21222,noscemus_756874,headed,,VERB,35,41,None,None
108428896,noscemus_756878_20567,noscemus_756878,Les,Les,PROPN,0,3,None,None
100898942,noscemus_888138_57149,noscemus_888138,corporis,corpus,NOUN,38,46,None,None
115204905,noscemus_744031_4973,noscemus_744031,",",",",PUNCT,84,85,None,None
106678555,noscemus_900765_6687,noscemus_900765,a,ab,ADP,88,89,None,None


In [17]:
sents_lagt, tokens_lagt = process_sentences_from_dir(lagt_sents_data_dir, "lagt")

In [ ]:
sents_cc, tokens_cc = process_sentences_from_dir(cc_sents_data_dir, "cc")

In [ ]:
# Merge
all_sents = pd.concat([sents_cc, sents_nos, sents_emlap, sents_lagt])
all_tokens = pd.concat([tokens_cc, tokens_nos, tokens_emlap, tokens_lagt])

## Add tokens and sentences into the database

In [ ]:
# add table with sentence data
conn.execute("CREATE TABLE IF NOT EXISTS sentences AS SELECT * FROM all_sents")

In [ ]:
# Store in batches if necessary:
for i, chunk in enumerate(np.array_split(all_tokens, 100)):
    conn.register("chunk", chunk)
    conn.execute("INSERT INTO tokens SELECT * FROM chunk")

In [ ]:
lemmata_df = all_tokens.groupby(["lemma", "pos"]).size().reset_index(name="count")
conn.execute("CREATE TABLE lemmata AS SELECT * FROM lemmata_df")